# Crisis Briefing Demo

Generates executive briefings from verified government data (USGS, NOAA, FEMA) using an LLM.

Run the cells top to bottom. Only the **Settings** cell needs editing.

- Your API key is read from the `.env` file in the project root (`ANTHROPIC_API_KEY=...`).
- Every briefing is **AI-generated from verified sources and must be reviewed by a person before public release.**
- Only active, current, non-superseded records are sent to the model. Everything the model returns is checked against the source text, and anything that fails the check is rejected.

## 1. Settings

In [1]:
# --- edit these -------------------------------------------------------------
REFRESH_DATA = False                   # True = pull fresh data from USGS/NOAA/FEMA first (free, ~10 s, saves files in data/live)
LLM = "anthropic:claude-haiku-5-5"    # cheap for testing; use "anthropic:claude-opus-5-5" for final results
MAX_EVENTS = 5                        # how many events to brief (each one is one LLM call)
PROMPT_VERSION = "v5"                 # prompt version from src/crisis/prompts.py ("v1" or "v2")
# -----------------------------------------------------------------------------

## 2. Setup

In [2]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.crisis.briefing import (DISCLOSURE, answer_question, build_briefing, check_consistency,
                                 get_llm, select_input_records)
from src.crisis.prompts import PROMPT_VERSIONS, TEMPLATES

llm = get_llm(LLM)
print("Project:", ROOT)
print("Model backend:", LLM, "| prompt version:", PROMPT_VERSION)

Project: c:\Users\cotto\Public-Catastrophic-Crisis-Briefing-Assistant
Model backend: anthropic:claude-haiku-5-5 | prompt version: v5


## 3. Load verified data

In [3]:
if REFRESH_DATA:
    from src.crisis.live_evidence import run_live_evidence_bundle
    bundle = run_live_evidence_bundle(ROOT)
    print("Fetched fresh data:", bundle["run_id"])
    for row in bundle["fetch_rows"]:
        print(f"  {row['source_name']}: HTTP {row['http_status']}, {row['record_count']} records")
    data_path = bundle["validated_output"]
else:
    data_path = sorted((ROOT / "data" / "live").glob("validated-output-*.json"))[-1]

batch = json.loads(Path(data_path).read_text(encoding="utf-8"))
usable = select_input_records(batch)
print(f"\nData file: {Path(data_path).name}")
print(f"{len(batch['records'])} verified records, {len(usable)} usable now (active, current, not superseded).")
if not usable:
    print("\nNothing is current. Set REFRESH_DATA = True and run again.")


Data file: validated-output-live_20261117_213530Z.json
24 verified records, 22 usable now (active, current, not superseded).


## 4. Generate briefings

In [4]:
briefing = build_briefing(batch, llm, max_events=MAX_EVENTS, version=PROMPT_VERSION)
print(f"{len(briefing['events'])} event(s) briefed with {briefing['model']}")

5 event(s) briefed with claude-haiku-5-5


## 5. Briefings

In [5]:

def show_briefing(b):
    print("=" * 78)
    print(f"EXECUTIVE BRIEFING   generated {b['generated_at_utc']}   model: {b['model']}")
    print(b['disclosure'])
    print("=" * 78)
    if not b["events"]:
        print("\nNo current verified information is available.")
    for n, e in enumerate(b["events"], 1):
        flag = "  [IMPORTANT: " + "; ".join(e["important"]["reasons"]) + "]" if e["important"]["is_important"] else ""
        print(f"\n{n}. {e['event_type'].upper()}  ({e['status']}){flag}")
        print(f"   {e['summary']}")
        f = e["facts"]
        if f["locations"]:
            print("   Locations:", "; ".join(dict.fromkeys(x["name"] for x in f["locations"])))
        if f["emergency_actions"]:
            print("   Emergency actions:")
            for x in f["emergency_actions"]:
                print("     -", x["action"])
        for field in ("casualties", "affected_population"):
            if f[field]:
                print(f"   {field.replace('_', ' ').title()}:", ", ".join(str(x["value"]) for x in f[field]))
        if e["not_reported"]:
            print("   Not reported by sources:", ", ".join(e["not_reported"]))
        for s in e["sources"][:3]:
            print(f"   Source: {s['source_name']} | published {s['published_at_utc']} | retrieved {s['retrieved_at_utc']}")
        if len(e["sources"]) > 3:
            print(f"   ... and {len(e['sources']) - 3} more sources")
        if e["conflicts"]["has_conflict"]:
            print("   CONFLICTING INFORMATION:", ", ".join(e["conflicts"]["flags"]))
            for d in e["conflicts"]["details"]:
                print(f"     - {d.get('field')}: sources report {d.get('values_seen')}")
        for note in e["review_notes"]:
            print("   REVIEW:", note)
    print()


show_briefing(briefing)

EXECUTIVE BRIEFING   generated 2026-11-17T22:47:35Z   model: claude-haiku-5-5
AI-generated from verified sources. A person must review this before public release.

1. STORM  (preliminary)  [IMPORTANT: NOAA alert update; sources conflict]
   NOAA reports a preliminary Small Craft Advisory for Point Piedras Blancas to Point Sal westward out to 10 NM, citing hazardous sea conditions that will be hazardous to small craft. Inexperienced mariners, especially those operating smaller vessels, should avoid navigating in the area. The records do not report any casualties or evacuations.
   Locations: Point Piedras Blancas to Point Sal westward out to 10 NM
   Emergency actions:
     - Inexperienced mariners, especially those operating smaller vessels, should avoid navigating in hazardous conditions.
   Not reported by sources: casualties, affected_population
   Source: NOAA National Weather Service Alerts | published 2026-11-17T21:30:00Z | retrieved 2026-11-17T21:35:30Z
   CONFLICTING INFORMATIO

## 6. Ask a question

Answers use only the same verified, current records. If the sources do not contain the answer, the assistant says so and lists what is missing instead of guessing.

In [6]:
def show_answer(res):
    print("Q:", res["question"])
    print("A:", res["answer"])
    if not res["answerable"]:
        print("   Not answerable from verified sources.")
    if res["information_gaps"]:
        print("   Information gaps:", "; ".join(res["information_gaps"]))
    for s in res.get("sources", [])[:3]:
        print(f"   Source: {s['source_name']} ({s['status']}), published {s['published_at_utc']}")
    if res["conflicts"]["has_conflict"]:
        print("   CONFLICTING INFORMATION:", ", ".join(res["conflicts"]["flags"]))
    for note in res["review_notes"]:
        print("   REVIEW:", note)
    if res["llm_error"]:
        print("   Note:", res["llm_error"])
    print("   " + res["disclosure"])
    print()


QUESTIONS = [
    {"template": "summary"},
    {"template": "casualties"},
    {"template": "emergency_actions"},
    {"question": "Which areas are affected?"},
]
for q in QUESTIONS:
    show_answer(answer_question(batch, q.get("question", ""), llm, template=q.get("template"), version=PROMPT_VERSION))

Q: Summarize the current situation in at most 3 sentences.
A: NOAA National Weather Service Los Angeles/Oxnard CA and Key West FL have issued Small Craft Advisories for hazardous sea conditions, and Key West FL reports East winds 20 to 25 kt with gusts up to 30 kt and seas in the Florida Straits 7 to 10 ft. The Pennsylvania Department of Environmental Protection has issued a code ORANGE air quality alert for the Mon Valley area, and USGS reports only small earthquakes (magnitudes 0.91 to 2) in Alaska and California. The records do not report casualties, affected population, or evacuations.
   Information gaps: No casualty figures are stated in the records.; No affected population figures are stated in the records.; No evacuation or shelter instructions are stated in the records.
   Source: USGS Earthquake Hazards Program (preliminary), published 2026-11-17T21:23:34Z
   Source: USGS Earthquake Hazards Program (preliminary), published 2026-11-17T21:22:13Z
   Source: NOAA National Weather

## 7. Consistency check (optional)

Asks the same question several times and reports whether the model gave the same kind of answer: whether it answered or declined, which numbers it stated, and which locations it extracted. `text_similarity` is wording overlap (1.0 = identical), for reference only. Costs one LLM call per run per question.

In [7]:
RUN_CONSISTENCY = True   # set True to run
CONSISTENCY_RUNS = 3

if RUN_CONSISTENCY:
    for q in [{"template": "summary"}, {"template": "casualties"}, {"template": "emergency_actions"}]:
        r = check_consistency(batch, q.get("question", ""), llm, runs=CONSISTENCY_RUNS,
                              template=q.get("template"), version=PROMPT_VERSION)
        print(f"{'CONSISTENT' if r['consistent'] else 'INCONSISTENT'}  {r['question']}")
        print(f"   answerable agree: {r['answerable_agree']} | numbers agree: {r['numbers_agree']} | "
              f"locations agree: {r['locations_agree']} | text similarity: {r['text_similarity']}")
        if not r["consistent"]:
            for i, a in enumerate(r["answers"], 1):
                print(f"   run {i}: {a}")
else:
    print("Skipped. Set RUN_CONSISTENCY = True to run.")

INCONSISTENT  Summarize the current situation in at most 3 sentences.
   answerable agree: True | numbers agree: False | locations agree: False | text similarity: 0.28
   run 1: Preliminary USGS data reports small earthquakes, all of magnitude 2 or less, including magnitude 1.6 near 29 km N of Skwentna, Alaska and magnitude 1.78 near 5 km NW of The Geysers, CA. NOAA National Weather Service has issued Small Craft Advisories for several coastal and offshore waters in California and Florida, advising that hazardous sea conditions will affect small craft. FEMA Open Data lists declaration 4955 (HURRICANE ISAIAS) for multiple Florida counties, and the records do not report evacuation or casualty information.
   run 2: USGS reports a series of preliminary small earthquakes, the largest at magnitude 2 near 6 km NNE of Willow, Alaska. NOAA has issued Small Craft Advisories for hazardous sea conditions in parts of the California coast and Florida waters, and an Air Quality Alert for the Mon Val

## 8. Compare prompt versions (optional)

Asks the fixed question set in `tests/prompt_questions.json` under each prompt version. Costs one LLM call per question per version.

In [8]:
RUN_COMPARISON = True   # set True to run

if RUN_COMPARISON:
    from tests.run_prompt_experiment import evaluate
    questions = json.loads((ROOT / "tests" / "prompt_questions.json").read_text(encoding="utf-8"))
    results = evaluate(batch, questions, llm, list(PROMPT_VERSIONS))
    print(f"{'version':<8}{'answered/declined correctly':>29}{'facts grounded':>16}{'fallback rate':>15}")
    for v, m in results.items():
        print(f"{v:<8}{m['answerable_match']:>29.0%}{m['fact_grounding_rate']:>16.0%}{m['fallback_rate']:>15.0%}")
    for v, m in results.items():
        print(f"\n--- {v} ---")
        for a in m["answers"]:
            print(f"Q: {a['question']}\nA: {a['answer']}")
            if a['answerable'] != a['expected']:
                print(f"   MISMATCH: model said answerable={a['answerable']}, the question set expects {a['expected']}")
            if a['error']:
                print(f"   WHY IT WAS NOT USED: {a['error']}")
            print()
else:
    print("Skipped. Set RUN_COMPARISON = True to compare prompt versions.")

version   answered/declined correctly  facts grounded  fallback rate
v1                               100%            100%             0%
v2                               100%            100%            12%
v3                               100%            100%             0%
v4                               100%            100%            12%
v5                               100%            100%             0%

--- v1 ---
Q: What weather alerts are currently active?
A: Preliminary NOAA records show several active weather alerts: Small Craft Advisories for multiple California coastal and Santa Barbara Channel waters, a Small Craft Advisory for Florida Bay and the Florida Straits, and a Code ORANGE Air Quality Alert for the Mon Valley area of Allegheny. The Air Quality Alert warns that sensitive groups such as children, the elderly, and people with asthma or heart or lung disease may be affected, and that avoiding outdoor exercise or strenuous activity can minimize effects.

Q: What eart